In [3]:
import pandas as pd
s3_bucket = "predictive-analytics-ml"
dataset_path = f"s3://{s3_bucket}/creditcard.csv"

In [4]:
df = pd.read_csv(dataset_path)
print(df.head())

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

In [3]:
pip install --upgrade s3fs aiobotocore

  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 109.2 MB/s  0:00:00
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.2.0
    Uninstalling fsspec-2026.2.0:
      Successfully uninstalled fsspec-2026.2.0
  Attempting uninstall: s3fs━━━━━━━━━━━━━╺━━━━━━ 10/12 [aiobotocore]]
    Found existing installation: s3fs 0.4.290m╺━━━━━━ 10/12 [aiobotocore]
    Uninstalling s3fs-0.4.2:━━━━━━━━━╺━━━━━━ 10/12 [aiobotocore]
      Successfully uninstalled s3fs-0.4.2╺━━━━━━ 10/12 [aiobotocore]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [s3fs]/12 [aiobotocore]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import s3fs
print(s3fs.__version__)  # should be 2023.x or newer

2026.3.0


In [5]:
df = df.dropna()

In [6]:
df.to_csv("processed_data.csv",index=False)

In [7]:
import sagemaker
from sagemaker.s3 import S3Uploader

sagemaker_session = sagemaker.Session()
s3_data_path = S3Uploader.upload("processed_data.csv",f"s3://{s3_bucket}/processed/")
print(f"data uploaded to {s3_data_path}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
data uploaded to s3://predictive-analytics-ml/processed//processed_data.csv


In [8]:
from sagemaker import get_execution_role
from sagemaker.amazon.linear_learner import LinearLearner

role = get_execution_role()
train_data = s3_data_path

In [9]:
linear = LinearLearner(role=role, instance_count = 1, instance_type = "ml.m4.xlarge", predictor_type = "binary_classifier", output_path = f"s3://{s3_bucket}/output/")

In [10]:
linear.fit({"train": train_data})

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 linear.fit({"train": train_data})                                                            │
│   2                                                                                              │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/workflow/pipeline_c │
│ ontext.py:346 in wrapper                                                                         │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/amazon/amazon_estim │
│ ator.py:263 in fit                                                                               │
│                                                                                                  │
│   260 │   │   │   │   will be unassociated.                                                      │
│   261 │   │   │   │   * `TrialComponentDisplayName` is used for display in Studio.               │
│   262 │   │   """                                                                                │
│ ❱ 263 │   │   self._prepare_for_training(records, job_name=job_name, mini_batch_size=mini_batc   │
│   264 │   │                                                                                      │
│   265 │   │   experiment_config = check_and_get_run_experiment_config(experiment_config)         │
│   266 │   │   self.latest_training_job = _TrainingJob.start_new(                                 │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/amazon/linear_learn │
│ er.py:438 in _prepare_for_training                                                               │
│                                                                                                  │
│   435 │   │   │   if num_records is None:                                                        │
│   436 │   │   │   │   raise ValueError("Must provide train channel.")                            │
│   437 │   │   else:                                                                              │
│ ❱ 438 │   │   │   num_records = records.num_records                                              │
│   439 │   │                                                                                      │
│   440 │   │   # mini_batch_size can't be greater than number of records or training job fails    │
│   441 │   │   mini_batch_size = mini_batch_size or self._get_default_mini_batch_size(num_recor   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
AttributeError: 'dict' object has no attribute 'num_records'

In [11]:
import numpy as np
from sagemaker import RecordSet

# Your features and labels as float32 numpy arrays
X_train = np.array(your_features, dtype=np.float32)
y_train = np.array(your_labels, dtype=np.float32)

# Convert to RecordSet (uploads to S3 automatically)
train_records = linear.record_set(X_train, labels=y_train, channel="train")

# Now fit using the RecordSet
linear.fit(train_records)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│    1 import numpy as np                                                                          │
│ ❱  2 from sagemaker import RecordSet                                                             │
│    3                                                                                             │
│    4 # Your features and labels as float32 numpy arrays                                          │
│    5 X_train = np.array(your_features, dtype=np.float32)                                         │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ImportError: cannot import name 'RecordSet' from 'sagemaker' 
(/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/__init__.py)

In [12]:
import numpy as np

# Make sure your data is float32 numpy arrays
X_train = np.array(X_train, dtype=np.float32)
y_train = np.array(y_train, dtype=np.float32)

# Convert to RecordSet — this uploads to S3 automatically
train_records = linear.record_set(X_train, labels=y_train, channel="train")

# Fit using the RecordSet object
linear.fit(train_records)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:4                                                                                    │
│                                                                                                  │
│    1 import numpy as np                                                                          │
│    2                                                                                             │
│    3 # Make sure your data is float32 numpy arrays                                               │
│ ❱  4 X_train = np.array(X_train, dtype=np.float32)                                               │
│    5 y_train = np.array(y_train, dtype=np.float32)                                               │
│    6                                                                                             │
│    7 # Convert to RecordSet — this uploads to S3 automatically                                   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'X_train' is not defined

In [13]:
from sagemaker.amazon.linear_learner import LinearLearner
import numpy as np

# 1. Define the estimator (your existing code — this is fine)
linear = LinearLearner(
    role=role,
    instance_count=1,
    instance_type="ml.m4.xlarge",
    predictor_type="binary_classifier",
    output_path=f"s3://{s3_bucket}/output/"
)

# 2. Prepare data as float32
X_train = np.array(X_train, dtype=np.float32)
y_train = np.array(y_train, dtype=np.float32)

# 3. Convert to RecordSet
train_records = linear.record_set(X_train, labels=y_train, channel="train")

# 4. Train
linear.fit(train_records)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:14                                                                                   │
│                                                                                                  │
│   11 )                                                                                           │
│   12                                                                                             │
│   13 # 2. Prepare data as float32                                                                │
│ ❱ 14 X_train = np.array(X_train, dtype=np.float32)                                               │
│   15 y_train = np.array(y_train, dtype=np.float32)                                               │
│   16                                                                                             │
│   17 # 3. Convert to RecordSet                                                                   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'X_train' is not defined

In [14]:
linear = LinearLearner(
    role=role,
    instance_count=1,
    instance_type="ml.m4.xlarge",
    predictor_type="binary_classifier",
    positive_example_weight_mult="balanced",  # handles class imbalance
    output_path=f"s3://{s3_bucket}/output/"
)

In [15]:
import numpy as np

# 1. Split into features and labels
X_train = df.drop(columns=["Class"]).values.astype(np.float32)
y_train = df["Class"].values.astype(np.float32)

# 2. Convert to RecordSet
train_records = linear.record_set(X_train, labels=y_train, channel="train")

# 3. Train the model
linear.fit(train_records)

INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating training-job with name: linear-learner-2026-03-31-10-21-46-057


2026-03-31 10:21:46 Starting - Starting the training job...
2026-03-31 10:22:07 Starting - Preparing the instances for training...
2026-03-31 10:22:30 Downloading - Downloading input data...
2026-03-31 10:23:00 Downloading - Downloading the training image......
2026-03-31 10:24:21 Training - Training image download completed. Training in progress....Docker entrypoint called with argument(s): train
Running default environment configuration script
[03/31/2026 10:24:39 INFO 140470076831552] Reading default configuration from /opt/amazon/lib/python3.8/site-packages/algorithm/resources/default-input.json: {'mini_batch_size': '1000', 'epochs': '15', 'feature_dim': 'auto', 'use_bias': 'true', 'binary_classifier_model_selection_criteria': 'accuracy', 'f_beta': '1.0', 'target_recall': '0.8', 'target_precision': '0.8', 'num_models': 'auto', 'num_calibration_samples': '10000000', 'init_method': 'uniform', 'init_scale': '0.07', 'init_sigma': '0.01', 'init_bias': '0.0', 'optimizer': 'auto', 'loss':

In [16]:
predictor = linear.deploy(instance_type = "ml.m4.xlarge", initial_instance_count = 1)

INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating model with name: linear-learner-2026-03-31-10-29-47-819
INFO:sagemaker:Creating endpoint-config with name linear-learner-2026-03-31-10-29-47-819
INFO:sagemaker:Creating endpoint with name linear-learner-2026-03-31-10-29-47-819


-------!

In [18]:
import numpy as np

test_sample = np.array([[0.5, 0.2, 0.8]])
prediction = predictor.predict(test_sample)
print("Prediction", prediction)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:4                                                                                    │
│                                                                                                  │
│   1 import numpy as np                                                                           │
│   2                                                                                              │
│   3 test_sample = np.array([[0.5, 0.2, 0.8]])                                                    │
│ ❱ 4 prediction = predictor.predict(test_sample)                                                  │
│   5 print("Prediction", prediction)                                                              │
│   6                                                                                              │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/base_predictor.py:2 │
│ 12 in predict                                                                                    │
│                                                                                                  │
│   209 │   │   if inference_component_name:                                                       │
│   210 │   │   │   request_args["InferenceComponentName"] = inference_component_name              │
│   211 │   │                                                                                      │
│ ❱ 212 │   │   response = self.sagemaker_session.sagemaker_runtime_client.invoke_endpoint(**req   │
│   213 │   │   return self._handle_response(response)                                             │
│   214 │                                                                                          │
│   215 │   def _handle_response(self, response):                                                  │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/botocore/client.py:602 in     │
│ _api_call                                                                                        │
│                                                                                                  │
│    599 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    600 │   │   │   │   )                                                                         │
│    601 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  602 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    603 │   │                                                                                     │
│    604 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    605                                                                                           │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/botocore/context.py:123 in    │
│ wrapper                                                                                          │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                

In [19]:
X_train = df.drop(columns=["Class", "Time"]).values.astype(np.float32)

In [20]:
import numpy as np
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

# Set serializer/deserializer
predictor.serializer = CSVSerializer()
predictor.deserializer = JSONDeserializer()

# Grab a few test samples (drop Class and Time columns)
X_test = df.drop(columns=["Class", "Time"]).values.astype(np.float32)
y_test = df["Class"].values

# Test on first 5 samples
sample = X_test[:5]
result = predictor.predict(sample)

# Print predictions vs actual
for i, pred in enumerate(result["predictions"]):
    print(f"Sample {i+1} | Predicted: {pred['predicted_label']} | Actual: {y_test[i]}")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:15                                                                                   │
│                                                                                                  │
│   12                                                                                             │
│   13 # Test on first 5 samples                                                                   │
│   14 sample = X_test[:5]                                                                         │
│ ❱ 15 result = predictor.predict(sample)                                                          │
│   16                                                                                             │
│   17 # Print predictions vs actual                                                               │
│   18 for i, pred in enumerate(result["predictions"]):                                            │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/base_predictor.py:2 │
│ 12 in predict                                                                                    │
│                                                                                                  │
│   209 │   │   if inference_component_name:                                                       │
│   210 │   │   │   request_args["InferenceComponentName"] = inference_component_name              │
│   211 │   │                                                                                      │
│ ❱ 212 │   │   response = self.sagemaker_session.sagemaker_runtime_client.invoke_endpoint(**req   │
│   213 │   │   return self._handle_response(response)                                             │
│   214 │                                                                                          │
│   215 │   def _handle_response(self, response):                                                  │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/botocore/client.py:602 in     │
│ _api_call                                                                                        │
│                                                                                                  │
│    599 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    600 │   │   │   │   )                                                                         │
│    601 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  602 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    603 │   │                                                                                     │
│    604 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    605                                                                                           │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/botocore/context.py:123 in    │
│ wrapper                                                                                          │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)         

In [21]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer
import numpy as np

# Set serializer/deserializer
predictor.serializer = CSVSerializer()
predictor.deserializer = JSONDeserializer()

# Grab test samples
X_test = df.drop(columns=["Class", "Time"]).values.astype(np.float32)
y_test = df["Class"].values

# ✅ Fix: convert to list before predicting
sample = X_test[:5].tolist()
result = predictor.predict(sample)

# Print predictions vs actual
for i, pred in enumerate(result["predictions"]):
    print(f"Sample {i+1} | Predicted: {pred['predicted_label']} | Actual: {y_test[i]}")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:15                                                                                   │
│                                                                                                  │
│   12                                                                                             │
│   13 # ✅ Fix: convert to list before predicting                                                 │
│   14 sample = X_test[:5].tolist()                                                                │
│ ❱ 15 result = predictor.predict(sample)                                                          │
│   16                                                                                             │
│   17 # Print predictions vs actual                                                               │
│   18 for i, pred in enumerate(result["predictions"]):                                            │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/base_predictor.py:2 │
│ 12 in predict                                                                                    │
│                                                                                                  │
│   209 │   │   if inference_component_name:                                                       │
│   210 │   │   │   request_args["InferenceComponentName"] = inference_component_name              │
│   211 │   │                                                                                      │
│ ❱ 212 │   │   response = self.sagemaker_session.sagemaker_runtime_client.invoke_endpoint(**req   │
│   213 │   │   return self._handle_response(response)                                             │
│   214 │                                                                                          │
│   215 │   def _handle_response(self, response):                                                  │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/botocore/client.py:602 in     │
│ _api_call                                                                                        │
│                                                                                                  │
│    599 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    600 │   │   │   │   )                                                                         │
│    601 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  602 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    603 │   │                                                                                     │
│    604 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    605                                                                                           │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/botocore/context.py:123 in    │
│ wrapper                                                                                          │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)          

In [22]:
# Test a single row first
single_sample = X_test[0].tolist()
result = predictor.predict(single_sample)
print(result)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│   1 # Test a single row first                                                                    │
│   2 single_sample = X_test[0].tolist()                                                           │
│ ❱ 3 result = predictor.predict(single_sample)                                                    │
│   4 print(result)                                                                                │
│   5                                                                                              │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/base_predictor.py:2 │
│ 12 in predict                                                                                    │
│                                                                                                  │
│   209 │   │   if inference_component_name:                                                       │
│   210 │   │   │   request_args["InferenceComponentName"] = inference_component_name              │
│   211 │   │                                                                                      │
│ ❱ 212 │   │   response = self.sagemaker_session.sagemaker_runtime_client.invoke_endpoint(**req   │
│   213 │   │   return self._handle_response(response)                                             │
│   214 │                                                                                          │
│   215 │   def _handle_response(self, response):                                                  │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/botocore/client.py:602 in     │
│ _api_call                                                                                        │
│                                                                                                  │
│    599 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    600 │   │   │   │   )                                                                         │
│    601 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  602 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    603 │   │                                                                                     │
│    604 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    605                                                                                           │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/botocore/context.py:123 in    │
│ wrapper                                                                                          │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                               

In [23]:
import boto3
client = boto3.client("sagemaker")
status = client.describe_endpoint(EndpointName=predictor.endpoint_name)
print(status["EndpointStatus"])  # should say "InService"

InService


In [24]:
import numpy as np
import json

# Grab test samples
X_test = df.drop(columns=["Class", "Time"]).values.astype(np.float32)
y_test = df["Class"].values

# Manually convert to CSV string
sample = X_test[:5]
csv_string = "\n".join([",".join(map(str, row)) for row in sample])

# Invoke endpoint directly via boto3
import boto3
runtime = boto3.client("sagemaker-runtime")

response = runtime.invoke_endpoint(
    EndpointName=predictor.endpoint_name,
    ContentType="text/csv",
    Body=csv_string
)

result = json.loads(response["Body"].read().decode("utf-8"))

# Print predictions vs actual
for i, pred in enumerate(result["predictions"]):
    print(f"Sample {i+1} | Predicted: {pred['predicted_label']} | Actual: {y_test[i]}")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:16                                                                                   │
│                                                                                                  │
│   13 import boto3                                                                                │
│   14 runtime = boto3.client("sagemaker-runtime")                                                 │
│   15                                                                                             │
│ ❱ 16 response = runtime.invoke_endpoint(                                                         │
│   17 │   EndpointName=predictor.endpoint_name,                                                   │
│   18 │   ContentType="text/csv",                                                                 │
│   19 │   Body=csv_string                                                                         │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/botocore/client.py:602 in     │
│ _api_call                                                                                        │
│                                                                                                  │
│    599 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    600 │   │   │   │   )                                                                         │
│    601 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  602 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    603 │   │                                                                                     │
│    604 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    605                                                                                           │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/botocore/context.py:123 in    │
│ wrapper                                                                                          │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/botocore/client.py:1078 in    │
│ _make_api_call                                                                                   │
│                                                                                                  │
│   1075 │   │   │   │   'error_code_override'                                                     │
│   1076 │   │   │   ) or error_info.get("Code")                                                   │
│   1077 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1078 │   │   │   raise error_class(parsed_response, opera

In [25]:
# Check what's being sent
print("Shape:", sample.shape)
print("CSV preview:\n", csv_string[:500])
print("Number of features:", sample.shape[1])

Shape: (5, 29)
CSV preview:
 -1.3598071,-0.072781175,2.5363467,1.3781552,-0.33832076,0.46238777,0.23959856,0.0986979,0.36378697,0.09079417,-0.55159956,-0.61780083,-0.9913899,-0.31116936,1.468177,-0.4704005,0.20797125,0.02579058,0.40399295,0.2514121,-0.018306779,0.27783757,-0.11047391,0.066928074,0.12853935,-0.18911484,0.13355838,-0.021053053,149.62
1.1918571,0.2661507,0.16648011,0.4481541,0.06001765,-0.08236081,-0.07880298,0.08510166,-0.25542513,-0.16697441,1.6127267,1.0652353,0.489095,-0.14377229,0.63555807,0.46391705,-0.1
Number of features: 29


In [26]:
# Include Time column — match exactly what the model was trained on
X_test = df.drop(columns=["Class"]).values.astype(np.float32)
y_test = df["Class"].values

sample = X_test[:5]
csv_string = "\n".join([",".join(map(str, row)) for row in sample])

response = runtime.invoke_endpoint(
    EndpointName=predictor.endpoint_name,
    ContentType="text/csv",
    Body=csv_string.encode("utf-8")
)

result = json.loads(response["Body"].read().decode("utf-8"))
for i, pred in enumerate(result["predictions"]):
    print(f"Sample {i+1} | Predicted: {pred['predicted_label']} | Actual: {y_test[i]}")

Sample 1 | Predicted: 0 | Actual: 0
Sample 2 | Predicted: 0 | Actual: 0
Sample 3 | Predicted: 0 | Actual: 0
Sample 4 | Predicted: 0 | Actual: 0
Sample 5 | Predicted: 0 | Actual: 0


In [27]:
from sklearn.metrics import classification_report, confusion_matrix
import json

X_test = df.drop(columns=["Class"]).values.astype(np.float32)
y_test = df["Class"].values

# Run predictions in batches of 100
y_pred = []
batch_size = 100

for i in range(0, len(X_test), batch_size):
    batch = X_test[i:i+batch_size]
    csv_string = "\n".join([",".join(map(str, row)) for row in batch])
    response = runtime.invoke_endpoint(
        EndpointName=predictor.endpoint_name,
        ContentType="text/csv",
        Body=csv_string.encode("utf-8")
    )
    result = json.loads(response["Body"].read().decode("utf-8"))
    y_pred.extend([p["predicted_label"] for p in result["predictions"]])

print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00    284315
       Fraud       0.80      0.80      0.80       492

    accuracy                           1.00    284807
   macro avg       0.90      0.90      0.90    284807
weighted avg       1.00      1.00      1.00    284807

Confusion Matrix:
[[284215    100]
 [    97    395]]


In [28]:
# Lower threshold = catch more fraud (higher recall, lower precision)
threshold = 0.4  # default is 0.5

y_pred_tuned = []
for p in all_predictions:
    y_pred_tuned.append(1.0 if p["score"] >= threshold else 0.0)

print(classification_report(y_test, y_pred_tuned, target_names=["Legitimate", "Fraud"]))

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:5                                                                                    │
│                                                                                                  │
│   2 threshold = 0.4  # default is 0.5                                                            │
│   3                                                                                              │
│   4 y_pred_tuned = []                                                                            │
│ ❱ 5 for p in all_predictions:                                                                    │
│   6 │   y_pred_tuned.append(1.0 if p["score"] >= threshold else 0.0)                             │
│   7                                                                                              │
│   8 print(classification_report(y_test, y_pred_tuned, target_names=["Legitimate", "Fraud"]))     │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'all_predictions' is not defined

In [29]:
from sklearn.metrics import classification_report, confusion_matrix
import json

X_test = df.drop(columns=["Class"]).values.astype(np.float32)
y_test = df["Class"].values

# Run predictions and store raw scores
all_predictions = []
batch_size = 100

for i in range(0, len(X_test), batch_size):
    batch = X_test[i:i+batch_size]
    csv_string = "\n".join([",".join(map(str, row)) for row in batch])
    response = runtime.invoke_endpoint(
        EndpointName=predictor.endpoint_name,
        ContentType="text/csv",
        Body=csv_string.encode("utf-8")
    )
    result = json.loads(response["Body"].read().decode("utf-8"))
    all_predictions.extend(result["predictions"])  # store full prediction objects

# Apply threshold
threshold = 0.4
y_pred_tuned = [1.0 if p["score"] >= threshold else 0.0 for p in all_predictions]

print(classification_report(y_test, y_pred_tuned, target_names=["Legitimate", "Fraud"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

              precision    recall  f1-score   support

  Legitimate       1.00      0.99      0.99    284315
       Fraud       0.10      0.91      0.18       492

    accuracy                           0.99    284807
   macro avg       0.55      0.95      0.59    284807
weighted avg       1.00      0.99      0.99    284807

Confusion Matrix:
[[280254   4061]
 [    45    447]]


In [30]:
from sklearn.metrics import classification_report, confusion_matrix

# Test multiple thresholds at once
thresholds = [0.45, 0.48, 0.50, 0.52, 0.55]

for threshold in thresholds:
    y_pred_tuned = [1.0 if p["score"] >= threshold else 0.0 for p in all_predictions]
    cm = confusion_matrix(y_test, y_pred_tuned)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    print(f"Threshold: {threshold} | Precision: {precision:.2f} | Recall: {recall:.2f} | F1: {f1:.2f} | Missed Fraud: {fn} | False Alarms: {fp}")

Threshold: 0.45 | Precision: 0.12 | Recall: 0.90 | F1: 0.21 | Missed Fraud: 47 | False Alarms: 3354
Threshold: 0.48 | Precision: 0.13 | Recall: 0.90 | F1: 0.22 | Missed Fraud: 47 | False Alarms: 3030
Threshold: 0.5 | Precision: 0.14 | Recall: 0.90 | F1: 0.24 | Missed Fraud: 48 | False Alarms: 2834
Threshold: 0.52 | Precision: 0.14 | Recall: 0.90 | F1: 0.25 | Missed Fraud: 48 | False Alarms: 2641
Threshold: 0.55 | Precision: 0.16 | Recall: 0.90 | F1: 0.27 | Missed Fraud: 50 | False Alarms: 2374


In [31]:
# 1. Delete current endpoint first (saves cost)
predictor.delete_endpoint()

# 2. Retrain without Time and with balanced weights
from sagemaker.amazon.linear_learner import LinearLearner

linear = LinearLearner(
    role=role,
    instance_count=1,
    instance_type="ml.m4.xlarge",
    predictor_type="binary_classifier",
    positive_example_weight_mult="balanced",  # key fix
    output_path=f"s3://{s3_bucket}/output/"
)

X_train = df.drop(columns=["Class", "Time"]).values.astype(np.float32)
y_train = df["Class"].values.astype(np.float32)

train_records = linear.record_set(X_train, labels=y_train, channel="train")
linear.fit(train_records)

# 3. Redeploy
predictor = linear.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge"
)

INFO:sagemaker:Deleting endpoint configuration with name: linear-learner-2026-03-31-10-29-47-819
INFO:sagemaker:Deleting endpoint with name: linear-learner-2026-03-31-10-29-47-819
INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating training-job with name: linear-learner-2026-03-31-11-03-00-489


2026-03-31 11:03:03 Starting - Starting the training job...
2026-03-31 11:03:17 Starting - Preparing the instances for training...
2026-03-31 11:03:43 Downloading - Downloading input data...
2026-03-31 11:04:14 Downloading - Downloading the training image.........
2026-03-31 11:05:40 Training - Training image download completed. Training in progress..Docker entrypoint called with argument(s): train
Running default environment configuration script
[03/31/2026 11:06:04 INFO 140489966659392] Reading default configuration from /opt/amazon/lib/python3.8/site-packages/algorithm/resources/default-input.json: {'mini_batch_size': '1000', 'epochs': '15', 'feature_dim': 'auto', 'use_bias': 'true', 'binary_classifier_model_selection_criteria': 'accuracy', 'f_beta': '1.0', 'target_recall': '0.8', 'target_precision': '0.8', 'num_models': 'auto', 'num_calibration_samples': '10000000', 'init_method': 'uniform', 'init_scale': '0.07', 'init_sigma': '0.01', 'init_bias': '0.0', 'optimizer': 'auto', 'loss'

INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating model with name: linear-learner-2026-03-31-11-08-49-523


Training seconds: 279
Billable seconds: 279


INFO:sagemaker:Creating endpoint-config with name linear-learner-2026-03-31-11-08-49-523
INFO:sagemaker:Creating endpoint with name linear-learner-2026-03-31-11-08-49-523


-------!

In [32]:
from sklearn.metrics import classification_report, confusion_matrix
import json

X_test = df.drop(columns=["Class", "Time"]).values.astype(np.float32)
y_test = df["Class"].values

all_predictions = []
for i in range(0, len(X_test), 100):
    csv_string = "\n".join([",".join(map(str, row)) for row in X_test[i:i+100]])
    response = runtime.invoke_endpoint(
        EndpointName=predictor.endpoint_name,
        ContentType="text/csv",
        Body=csv_string.encode("utf-8")
    )
    all_predictions.extend(json.loads(response["Body"].read())["predictions"])

y_pred = [1.0 if p["score"] >= 0.5 else 0.0 for p in all_predictions]
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

  Legitimate       1.00      0.99      1.00    284315
       Fraud       0.17      0.89      0.28       492

    accuracy                           0.99    284807
   macro avg       0.58      0.94      0.64    284807
weighted avg       1.00      0.99      0.99    284807

[[282096   2219]
 [    53    439]]


In [33]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Prepare data
X = df.drop(columns=["Class", "Time"]).values.astype(np.float32)
y = df["Class"].values.astype(np.float32)

# 2. Split FIRST before SMOTE
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
print(f"Train fraud cases: {sum(y_train==1)}, Test fraud cases: {sum(y_test==1)}")

# 3. Apply SMOTE only on training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE:")
print(f"Train Legitimate: {sum(y_train_resampled==0)}, Train Fraud: {sum(y_train_resampled==1)}")

# 4. Retrain
linear = LinearLearner(
    role=role,
    instance_count=1,
    instance_type="ml.m4.xlarge",
    predictor_type="binary_classifier",
    output_path=f"s3://{s3_bucket}/output/"
)

train_records = linear.record_set(
    X_train_resampled.astype(np.float32),
    labels=y_train_resampled.astype(np.float32),
    channel="train"
)
linear.fit(train_records)

# 5. Redeploy
predictor = linear.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge"
)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱  1 from imblearn.over_sampling import SMOTE                                                    │
│    2 from sklearn.model_selection import train_test_split                                        │
│    3 import numpy as np                                                                          │
│    4                                                                                             │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ModuleNotFoundError: No module named 'imblearn'

In [34]:
from imblearn.over_sampling import SMOTE
import numpy as np

# 1. Delete current endpoint
predictor.delete_endpoint()

# 2. Prepare data
X = df.drop(columns=["Class", "Time"]).values.astype(np.float32)
y = df["Class"].values.astype(np.float32)

print(f"Before SMOTE: Legitimate={sum(y==0)}, Fraud={sum(y==1)}")

# 3. Apply SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print(f"After SMOTE: Legitimate={sum(y_resampled==0)}, Fraud={sum(y_resampled==1)}")

# 4. Retrain without positive_example_weight_mult
linear = LinearLearner(
    role=role,
    instance_count=1,
    instance_type="ml.m4.xlarge",
    predictor_type="binary_classifier",
    output_path=f"s3://{s3_bucket}/output/"
)

train_records = linear.record_set(
    X_resampled.astype(np.float32),
    labels=y_resampled.astype(np.float32),
    channel="train"
)
linear.fit(train_records)

# 5. Redeploy
predictor = linear.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge"
)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱  1 from imblearn.over_sampling import SMOTE                                                    │
│    2 import numpy as np                                                                          │
│    3                                                                                             │
│    4 # 1. Delete current endpoint                                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ModuleNotFoundError: No module named 'imblearn'

In [35]:
!pip install imbalanced-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [imbalanced-learn][imbalanced-learn]


In [36]:
from imblearn.over_sampling import SMOTE
import numpy as np

# 1. Delete current endpoint
predictor.delete_endpoint()

# 2. Prepare data
X = df.drop(columns=["Class", "Time"]).values.astype(np.float32)
y = df["Class"].values.astype(np.float32)

print(f"Before SMOTE: Legitimate={sum(y==0)}, Fraud={sum(y==1)}")

# 3. Apply SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print(f"After SMOTE: Legitimate={sum(y_resampled==0)}, Fraud={sum(y_resampled==1)}")

# 4. Retrain without positive_example_weight_mult
linear = LinearLearner(
    role=role,
    instance_count=1,
    instance_type="ml.m4.xlarge",
    predictor_type="binary_classifier",
    output_path=f"s3://{s3_bucket}/output/"
)

train_records = linear.record_set(
    X_resampled.astype(np.float32),
    labels=y_resampled.astype(np.float32),
    channel="train"
)
linear.fit(train_records)

# 5. Redeploy
predictor = linear.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge"
)

INFO:sagemaker:Deleting endpoint configuration with name: linear-learner-2026-03-31-11-08-49-523
INFO:sagemaker:Deleting endpoint with name: linear-learner-2026-03-31-11-08-49-523


Before SMOTE: Legitimate=284315, Fraud=492
After SMOTE: Legitimate=284315, Fraud=284315


INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating training-job with name: linear-learner-2026-03-31-11-25-10-114


2026-03-31 11:25:12 Starting - Starting the training job...
2026-03-31 11:25:26 Starting - Preparing the instances for training...
2026-03-31 11:25:48 Downloading - Downloading input data...
2026-03-31 11:26:18 Downloading - Downloading the training image......
2026-03-31 11:27:39 Training - Training image download completed. Training in progress...Docker entrypoint called with argument(s): train
Running default environment configuration script
[03/31/2026 11:27:55 INFO 140598875965248] Reading default configuration from /opt/amazon/lib/python3.8/site-packages/algorithm/resources/default-input.json: {'mini_batch_size': '1000', 'epochs': '15', 'feature_dim': 'auto', 'use_bias': 'true', 'binary_classifier_model_selection_criteria': 'accuracy', 'f_beta': '1.0', 'target_recall': '0.8', 'target_precision': '0.8', 'num_models': 'auto', 'num_calibration_samples': '10000000', 'init_method': 'uniform', 'init_scale': '0.07', 'init_sigma': '0.01', 'init_bias': '0.0', 'optimizer': 'auto', 'loss': 

INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating model with name: linear-learner-2026-03-31-11-29-27-947


Training seconds: 205
Billable seconds: 205


INFO:sagemaker:Creating endpoint-config with name linear-learner-2026-03-31-11-29-27-947
INFO:sagemaker:Creating endpoint with name linear-learner-2026-03-31-11-29-27-947


-------!

In [37]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import json

# Use original df for clean test set (no SMOTE)
X = df.drop(columns=["Class", "Time"]).values.astype(np.float32)
y = df["Class"].values.astype(np.float32)

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Evaluate
all_predictions = []
for i in range(0, len(X_test), 100):
    csv_string = "\n".join([",".join(map(str, row)) for row in X_test[i:i+100]])
    response = runtime.invoke_endpoint(
        EndpointName=predictor.endpoint_name,
        ContentType="text/csv",
        Body=csv_string.encode("utf-8")
    )
    all_predictions.extend(json.loads(response["Body"].read())["predictions"])

y_pred = [1.0 if p["score"] >= 0.5 else 0.0 for p in all_predictions]
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

  Legitimate       0.00      0.00      0.00     56864
       Fraud       0.00      1.00      0.00        98

    accuracy                           0.00     56962
   macro avg       0.00      0.50      0.00     56962
weighted avg       0.00      0.00      0.00     56962

[[    0 56864]
 [    0    98]]


/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"

In [38]:
predictor.delete_endpoint()


INFO:sagemaker:Deleting endpoint configuration with name: linear-learner-2026-03-31-11-29-27-947
INFO:sagemaker:Deleting endpoint with name: linear-learner-2026-03-31-11-29-27-947


In [39]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Class", "Time"]).values.astype(np.float32)
y = df["Class"].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train fraud: {sum(y_train==1)}, Test fraud: {sum(y_test==1)}")

Train fraud: 394, Test fraud: 98


In [40]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print(f"After SMOTE - Legitimate: {sum(y_train_resampled==0)}, Fraud: {sum(y_train_resampled==1)}")

After SMOTE - Legitimate: 227451, Fraud: 227451


In [41]:
linear = LinearLearner(
    role=role,
    instance_count=1,
    instance_type="ml.m4.xlarge",
    predictor_type="binary_classifier",
    output_path=f"s3://{s3_bucket}/output/"
)

train_records = linear.record_set(
    X_train_resampled.astype(np.float32),
    labels=y_train_resampled.astype(np.float32),
    channel="train"
)
linear.fit(train_records)

INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating training-job with name: linear-learner-2026-03-31-11-39-48-791


2026-03-31 11:39:50 Starting - Starting the training job...
2026-03-31 11:40:05 Starting - Preparing the instances for training...
2026-03-31 11:40:28 Downloading - Downloading input data...
2026-03-31 11:40:59 Downloading - Downloading the training image......
2026-03-31 11:42:20 Training - Training image download completed. Training in progress.....Docker entrypoint called with argument(s): train
Running default environment configuration script
[03/31/2026 11:42:50 INFO 139666867337024] Reading default configuration from /opt/amazon/lib/python3.8/site-packages/algorithm/resources/default-input.json: {'mini_batch_size': '1000', 'epochs': '15', 'feature_dim': 'auto', 'use_bias': 'true', 'binary_classifier_model_selection_criteria': 'accuracy', 'f_beta': '1.0', 'target_recall': '0.8', 'target_precision': '0.8', 'num_models': 'auto', 'num_calibration_samples': '10000000', 'init_method': 'uniform', 'init_scale': '0.07', 'init_sigma': '0.01', 'init_bias': '0.0', 'optimizer': 'auto', 'loss'

In [42]:
from sklearn.metrics import classification_report, confusion_matrix
import json

all_predictions = []
for i in range(0, len(X_test), 100):
    csv_string = "\n".join([",".join(map(str, row)) for row in X_test[i:i+100]])
    response = runtime.invoke_endpoint(
        EndpointName=predictor.endpoint_name,
        ContentType="text/csv",
        Body=csv_string.encode("utf-8")
    )
    all_predictions.extend(json.loads(response["Body"].read())["predictions"])

y_pred = [1.0 if p["score"] >= 0.5 else 0.0 for p in all_predictions]
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))
print(confusion_matrix(y_test, y_pred))

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:7                                                                                    │
│                                                                                                  │
│    4 all_predictions = []                                                                        │
│    5 for i in range(0, len(X_test), 100):                                                        │
│    6 │   csv_string = "\n".join([",".join(map(str, row)) for row in X_test[i:i+100]])            │
│ ❱  7 │   response = runtime.invoke_endpoint(                                                     │
│    8 │   │   EndpointName=predictor.endpoint_name,                                               │
│    9 │   │   ContentType="text/csv",                                                             │
│   10 │   │   Body=csv_string.encode("utf-8")                                                     │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/botocore/client.py:602 in     │
│ _api_call                                                                                        │
│                                                                                                  │
│    599 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    600 │   │   │   │   )                                                                         │
│    601 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  602 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    603 │   │                                                                                     │
│    604 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    605                                                                                           │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/botocore/context.py:123 in    │
│ wrapper                                                                                          │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/botocore/client.py:1078 in    │
│ _make_api_call                                                                                   │
│                                                                                                  │
│   1075 │   │   │   │   'error_code_override'                                                     │
│   1076 │   │   │   ) or error_info.get("Code")                                                   │
│   1077 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1078 │   │   │   raise error_class(parsed_response, opera

In [43]:
predictor = linear.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge"
)

INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating model with name: linear-learner-2026-03-31-11-53-49-842
INFO:sagemaker:Creating endpoint-config with name linear-learner-2026-03-31-11-53-49-842
INFO:sagemaker:Creating endpoint with name linear-learner-2026-03-31-11-53-49-842


-------!

In [44]:
import boto3

client = boto3.client("sagemaker")
status = client.describe_endpoint(EndpointName=predictor.endpoint_name)
print(f"Endpoint: {status['EndpointName']}")
print(f"Status: {status['EndpointStatus']}")

Endpoint: linear-learner-2026-03-31-11-53-49-842
Status: InService


In [45]:
from sklearn.metrics import classification_report, confusion_matrix
import json

all_predictions = []
for i in range(0, len(X_test), 100):
    csv_string = "\n".join([",".join(map(str, row)) for row in X_test[i:i+100]])
    response = runtime.invoke_endpoint(
        EndpointName=predictor.endpoint_name,
        ContentType="text/csv",
        Body=csv_string.encode("utf-8")
    )
    all_predictions.extend(json.loads(response["Body"].read())["predictions"])

y_pred = [1.0 if p["score"] >= 0.5 else 0.0 for p in all_predictions]
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

  Legitimate       1.00      0.18      0.31     56864
       Fraud       0.00      0.99      0.00        98

    accuracy                           0.18     56962
   macro avg       0.50      0.59      0.16     56962
weighted avg       1.00      0.18      0.31     56962

[[10274 46590]
 [    1    97]]


In [46]:
from imblearn.over_sampling import SMOTE

# Instead of matching 100% (492 → 284315), only oversample to 10x fraud cases
smote = SMOTE(sampling_strategy=0.1, random_state=42)  # fraud will be 10% of legitimate
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print(f"After SMOTE - Legitimate: {sum(y_train_resampled==0)}, Fraud: {sum(y_train_resampled==1)}")

After SMOTE - Legitimate: 227451, Fraud: 22745


In [47]:
linear = LinearLearner(
    role=role,
    instance_count=1,
    instance_type="ml.m4.xlarge",
    predictor_type="binary_classifier",
    output_path=f"s3://{s3_bucket}/output/"
)

train_records = linear.record_set(
    X_train_resampled.astype(np.float32),
    labels=y_train_resampled.astype(np.float32),
    channel="train"
)
linear.fit(train_records)

INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating training-job with name: linear-learner-2026-03-31-12-01-35-844


2026-03-31 12:01:36 Starting - Starting the training job......
2026-03-31 12:02:36 Downloading - Downloading input data...
2026-03-31 12:03:06 Downloading - Downloading the training image.........
2026-03-31 12:04:32 Training - Training image download completed. Training in progress...Docker entrypoint called with argument(s): train
Running default environment configuration script
[03/31/2026 12:04:54 INFO 139850853607232] Reading default configuration from /opt/amazon/lib/python3.8/site-packages/algorithm/resources/default-input.json: {'mini_batch_size': '1000', 'epochs': '15', 'feature_dim': 'auto', 'use_bias': 'true', 'binary_classifier_model_selection_criteria': 'accuracy', 'f_beta': '1.0', 'target_recall': '0.8', 'target_precision': '0.8', 'num_models': 'auto', 'num_calibration_samples': '10000000', 'init_method': 'uniform', 'init_scale': '0.07', 'init_sigma': '0.01', 'init_bias': '0.0', 'optimizer': 'auto', 'loss': 'auto', 'margin': '1.0', 'quantile': '0.5', 'loss_insensitivity':

In [ ]:
predictor = linear.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge"
)

INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating model with name: linear-learner-2026-03-31-12-10-32-319
INFO:sagemaker:Creating endpoint-config with name linear-learner-2026-03-31-12-10-32-319
INFO:sagemaker:Creating endpoint with name linear-learner-2026-03-31-12-10-32-319


---------!

In [49]:
from sklearn.metrics import classification_report, confusion_matrix
import json

all_predictions = []
for i in range(0, len(X_test), 100):
    csv_string = "\n".join([",".join(map(str, row)) for row in X_test[i:i+100]])
    response = runtime.invoke_endpoint(
        EndpointName=predictor.endpoint_name,
        ContentType="text/csv",
        Body=csv_string.encode("utf-8")
    )
    all_predictions.extend(json.loads(response["Body"].read())["predictions"])

y_pred = [1.0 if p["score"] >= 0.5 else 0.0 for p in all_predictions]
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

  Legitimate       1.00      0.37      0.54     56864
       Fraud       0.00      0.98      0.01        98

    accuracy                           0.37     56962
   macro avg       0.50      0.68      0.27     56962
weighted avg       1.00      0.37      0.54     56962

[[21173 35691]
 [    2    96]]


In [50]:
from sklearn.metrics import classification_report, confusion_matrix
import json

all_predictions = []
for i in range(0, len(X_test), 100):
    csv_string = "\n".join([",".join(map(str, row)) for row in X_test[i:i+100]])
    response = runtime.invoke_endpoint(
        EndpointName=predictor.endpoint_name,
        ContentType="text/csv",
        Body=csv_string.encode("utf-8")
    )
    all_predictions.extend(json.loads(response["Body"].read())["predictions"])

y_pred = [1.0 if p["score"] >= 0.5 else 0.0 for p in all_predictions]
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

  Legitimate       1.00      0.37      0.54     56864
       Fraud       0.00      0.98      0.01        98

    accuracy                           0.37     56962
   macro avg       0.50      0.68      0.27     56962
weighted avg       1.00      0.37      0.54     56962

[[21173 35691]
 [    2    96]]


In [51]:
predictor.delete_endpoint()

INFO:sagemaker:Deleting endpoint configuration with name: linear-learner-2026-03-31-12-10-32-319
INFO:sagemaker:Deleting endpoint with name: linear-learner-2026-03-31-12-10-32-319


In [52]:
from sagemaker import image_uris
from sagemaker.estimator import Estimator

container = image_uris.retrieve("xgboost", sagemaker_session.boto_region_name, version="1.5-1")

xgb = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m4.xlarge",
    output_path=f"s3://{s3_bucket}/output/"
)

xgb.set_hyperparameters(
    objective="binary:logistic",
    num_round=100,
    max_depth=5,
    eta=0.2,
    scale_pos_weight=sum(y_train==0) / sum(y_train==1)
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [53]:
import pandas as pd
from sagemaker.s3 import S3Uploader

train_df = pd.DataFrame(X_train)
train_df.insert(0, "label", y_train)
train_df.to_csv("xgb_train.csv", index=False, header=False)

s3_train = S3Uploader.upload("xgb_train.csv", f"s3://{s3_bucket}/xgb/train/")
print(f"Training data uploaded to {s3_train}")

Training data uploaded to s3://predictive-analytics-ml/xgb/train//xgb_train.csv


In [54]:
from sagemaker.inputs import TrainingInput

xgb.fit({"train": TrainingInput(s3_train, content_type="text/csv")})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-03-31-12-24-28-286


2026-03-31 12:24:28 Starting - Starting the training job...
2026-03-31 12:25:02 Starting - Preparing the instances for training...
2026-03-31 12:25:29 Downloading - Downloading input data...
2026-03-31 12:26:00 Failed - Training job failed
..

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│   1 from sagemaker.inputs import TrainingInput                                                   │
│   2                                                                                              │
│ ❱ 3 xgb.fit({"train": TrainingInput(s3_train, content_type="text/csv")})                         │
│   4                                                                                              │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/telemetry/telemetry │
│ _logging.py:171 in wrapper                                                                       │
│                                                                                                  │
│   168 │   │   │   │   │   caught_ex = e                                                          │
│   169 │   │   │   │   finally:                                                                   │
│   170 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 171 │   │   │   │   │   │   raise caught_ex                                                    │
│   172 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   173 │   │   │   else:                                                                          │
│   174 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/telemetry/telemetry │
│ _logging.py:142 in wrapper                                                                       │
│                                                                                                  │
│   139 │   │   │   │   start_timer = perf_counter()                                               │
│   140 │   │   │   │   try:                                                                       │
│   141 │   │   │   │   │   # Call the original function                                           │
│ ❱ 142 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   143 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   144 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   145 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/workflow/pipeline_c │
│ ontext.py:346 in wrapper                                                                         │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                      

In [55]:
import pandas as pd
from sagemaker.s3 import S3Uploader

train_df = pd.DataFrame(X_train)
train_df.insert(0, "label", y_train)
train_df.to_csv("xgb_train.csv", index=False, header=False)

# Fix: removed trailing slash from path
s3_train = S3Uploader.upload("xgb_train.csv", f"s3://{s3_bucket}/xgb/train")
print(f"Training data uploaded to {s3_train}")

Training data uploaded to s3://predictive-analytics-ml/xgb/train/xgb_train.csv


In [56]:
from sagemaker.inputs import TrainingInput

xgb.fit({"train": TrainingInput(s3_train, content_type="text/csv")})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-03-31-12-29-23-955


2026-03-31 12:29:28 Starting - Starting the training job...
2026-03-31 12:29:42 Starting - Preparing the instances for training...
2026-03-31 12:30:05 Downloading - Downloading input data...
2026-03-31 12:30:35 Downloading - Downloading the training image......
2026-03-31 12:31:32 Training - Training image download completed. Training in progress./miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2026-03-31 12:31:49.361 ip-10-0-177-66.ec2.internal:6 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-03-31 12:31:49.382 ip-10-0-177-66.ec2.internal:6 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-03-31:12:31:49:INFO] Imported framework sagemaker_xgboost_container.training
[2026-03-31:12:31:49:INFO] Failed to parse hyperparameter objective value binary:

In [57]:
predictor = xgb.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge"
)

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-03-31-12-33-51-731
INFO:sagemaker:Creating endpoint-config with name sagemaker-xgboost-2026-03-31-12-33-51-731
INFO:sagemaker:Creating endpoint with name sagemaker-xgboost-2026-03-31-12-33-51-731


------!

In [58]:
client = boto3.client("sagemaker")
status = client.describe_endpoint(EndpointName=predictor.endpoint_name)
print(f"Status: {status['EndpointStatus']}")

Status: InService


In [59]:
from sklearn.metrics import classification_report, confusion_matrix
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer
import numpy as np
import json

# Set serializer
predictor.serializer = CSVSerializer()
predictor.deserializer = JSONDeserializer()

all_predictions = []
for i in range(0, len(X_test), 100):
    batch = X_test[i:i+100]
    csv_string = "\n".join([",".join(map(str, row)) for row in batch])
    response = runtime.invoke_endpoint(
        EndpointName=predictor.endpoint_name,
        ContentType="text/csv",
        Body=csv_string.encode("utf-8")
    )
    result = response["Body"].read().decode("utf-8")
    scores = [float(s) for s in result.strip().split("\n")]
    all_predictions.extend(scores)

# Apply threshold
y_pred = [1.0 if score >= 0.5 else 0.0 for score in all_predictions]
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     56864
       Fraud       0.83      0.86      0.84        98

    accuracy                           1.00     56962
   macro avg       0.92      0.93      0.92     56962
weighted avg       1.00      1.00      1.00     56962

[[56847    17]
 [   14    84]]


In [60]:
!pip install gitpython

  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.3-py3-none-any.whl.metadata (4.6 kB)
Using cached gitpython-3.1.46-py3-none-any.whl (208 kB)
Using cached gitdb-4.0.12-py3-none-any.whl (62 kB)
Using cached smmap-5.0.3-py3-none-any.whl (24 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [gitpython]/3 [gitpython]


In [61]:
cd /home/ec2-user/SageMaker

/home/ec2-user/SageMaker


In [62]:
git init

╭──────────────────────────────────────────────────────────────────────────────────────────────────╮
│ git init                                                                                         │
│     ▲                                                                                            │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
SyntaxError: invalid syntax

In [63]:
!git in it

git: 'in' is not a git command. See 'git --help'.

The most similar commands are
	init
	am
	diff
	gc
	mv
	rm


In [64]:
git init

╭──────────────────────────────────────────────────────────────────────────────────────────────────╮
│ git init                                                                                         │
│     ▲                                                                                            │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
SyntaxError: invalid syntax

In [65]:
git --help

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 git --help                                                                                   │
│   2                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'git' is not defined